## Libraries

In [ ]:
!pip install torch_geometric
!pip install torch-spline-conv


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os.path as osp
import torch
import torch.nn.functional as F
import torch.nn as nn
import time
import copy

import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SplineConv
from torch_geometric.typing import WITH_TORCH_SPLINE_CONV

import torch.nn.utils.prune as prune
from torch.nn.utils.prune import global_unstructured, L1Unstructured

In [2]:
def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=True) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

  

## Loading Dataset

In [3]:

if not WITH_TORCH_SPLINE_CONV:
    quit("This example requires 'torch-spline-conv'")

dataset = 'Cora'
transform = T.Compose([
    T.RandomNodeSplit(num_val=500, num_test=500),
    T.TargetIndegree(),
])
path =  'data'
dataset = Planetoid(path, dataset, transform=transform)
data = dataset[0]

## Model

In [4]:


class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SplineConv(dataset.num_features, 16, dim=1, kernel_size=2)
        self.conv2 = SplineConv(16, dataset.num_classes, dim=1, kernel_size=2)

    def forward(self):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        x = F.dropout(x, training=self.training)
        x = F.elu(self.conv1(x, edge_index, edge_attr))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index, edge_attr)
        return F.log_softmax(x, dim=1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, data = Net().to(device), data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)

In [5]:


def train(callbacks = None):
    model.train()
    optimizer.zero_grad()
    F.nll_loss(model()[data.train_mask], data.y[data.train_mask]).backward()
    optimizer.step()
    if callbacks is not None:
        for callback in callbacks:
                callback()


@torch.no_grad()
def test(model):
    model.eval()
    log_probs, accs = model(), []
    for _, mask in data('train_mask', 'test_mask'):
        pred = log_probs[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs




In [ ]:
for epoch in range(10):
    train()
    train_acc, test_acc = test(model)
    #if epoch % 10 == 0:
    print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Test: {test_acc:.4f}')

In [8]:
best_checkpoint = dict()
best_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
model.load_state_dict(best_checkpoint['state_dict'])
recover_model = lambda: model.load_state_dict(best_checkpoint['state_dict'])

In [9]:
t0=time.time()
train_acc, base_model_accuracy=test(model)
t1=time.time()
t_base_model=t1 - t0
###
base_model_size = get_model_size(model, count_nonzero_only=True)
num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"base model has accuracy on test set={base_model_accuracy:.2f}%")
print(f"base model has size={base_model_size/MiB:.2f} MiB")
print(f"The time inference of base model is ={t_base_model}") 
print(f"The number of parametrs of base model is:{num_parm_base_model}")

base model has accuracy on test set=0.87%
base model has size=0.26 MiB
The time inference of base model is =16.413111686706543
The number of parametrs of base model is:69143


# Let's Prune the Model and Re-Evaluate the Accuracy.
## Gobal Pruning

In [36]:
model

Net(
  (conv1): SplineConv(1433, 16, dim=1)
  (conv2): SplineConv(16, 7, dim=1)
)

In [38]:
for name, param in model.named_parameters():
  
        print(name)

conv1.weight
conv1.bias
conv1.lin.weight
conv2.weight
conv2.bias
conv2.lin.weight


In [7]:
def global_L1(model,parameters_to_prune, sparsity):
        global_unstructured(
        parameters_to_prune,
        pruning_method= L1Unstructured,
        amount=sparsity, )  
        
  
        for module in parameters_to_prune:
           prune.remove(module[0],'weight')


def get_model_sparsity(model: nn.Module) -> float:
        Sparsity=dict()
        global_zero=0
        global_nzero=0
        layyers=[]
        spars=[]
        for name, param in model.named_parameters(): 
            if 'weight' in name:
                        zero=float(torch.sum(param == 0))
                        nzero=float(param.nelement())
                        sparsity=  float(zero)/ float(nzero)
                        print( f'Sparsity in {name}: {sparsity:.3f}' )
                        layyers.append(name)
                        spars.append(sparsity)
                        global_zero +=zero
                        global_nzero +=nzero
                        


        Sparsity={key: value for key, value in zip(layyers,spars)}
        global_sparsity= float(global_zero) /float(global_nzero)
        Sparsity.update({'Global sparsity':  global_sparsity})
        print("Global sparsity: {:.3f}".format(global_sparsity))
        return   Sparsity   

In [11]:


sparsity =0.9
parameters_to_prune = [
    (model.conv1, 'weight'),
     (model.conv1.lin, 'weight'),
    (model.conv2, 'weight'),
     (model.conv2.lin, 'weight'),
]

In [12]:
global_L1(model,parameters_to_prune,  sparsity)

get_model_sparsity(model)

Sparsity in conv1.weight: 0.922
Sparsity in conv1.lin.weight: 0.864
Sparsity in conv2.weight: 0.473
Sparsity in conv2.lin.weight: 0.179
Global sparsity: 0.900


{'conv1.weight': 0.9217986741102582,
 'conv1.lin.weight': 0.8640963014654571,
 'conv2.weight': 0.4732142857142857,
 'conv2.lin.weight': 0.17857142857142858,
 'Global sparsity': 0.9}

In [13]:
t0=time.time()
train_acc, pruned_model_accuracy=test(model)
t1=time.time()
t_pruned_model=t1 - t0
###
pruned_model_size = get_model_size(model)
num_parm_pruned_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_model}")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB base model")

90.0% sparse model has accuracy on test set=0.85%
90.0% sparse model has size=0.03 MiB
The time inference of 90.0% sparse model is =19.087051153182983
The number of parametrs of 90.0% sparse model is:6935
90.0% sparse model has size=0.03 MiB, which is 9.97X smaller than the 0.26 MiB base model


## Finetuning Fine-grained Pruned Sparse Model

In [14]:

best_sparse_checkpoint = dict()
best_sparse_accuracy = 0
num_finetune_epochs=15
print(f'Finetuning Fine-grained Pruned Sparse Model')
for epoch in range(num_finetune_epochs):
    # At the end of each train iteration, we have to apply the pruning mask
    #    to keep the model sparse during the training
    train(callbacks=[lambda: global_L1(model,parameters_to_prune, sparsity)])
    train_acc, accuracy = test(model)
  
    is_best = accuracy > best_sparse_accuracy
    if is_best:
        best_sparse_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
        best_sparse_accuracy = accuracy
    
    #if epoch % 10 == 0:
    print(f'Epoch {epoch} Sparse Accuracy {accuracy:.2f}% / Best Sparse Accuracy: {best_sparse_accuracy:.2f}%')

            
model.load_state_dict(best_sparse_checkpoint['state_dict'])

t0=time.time()
train_acc, pruned_finetune_model_accuracy=test(model)
t1=time.time()
t_pruned_finetune_model=t1 - t0
###
pruned_finetune_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")            

Finetuning Fine-grained Pruned Sparse Model
    Epoch 0 Sparse Accuracy 0.84% / Best Sparse Accuracy: 0.84%
    Epoch 1 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 2 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 3 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 4 Sparse Accuracy 0.84% / Best Sparse Accuracy: 0.85%
    Epoch 5 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 6 Sparse Accuracy 0.84% / Best Sparse Accuracy: 0.85%
    Epoch 7 Sparse Accuracy 0.84% / Best Sparse Accuracy: 0.85%
    Epoch 8 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 9 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 10 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 11 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 12 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
    Epoch 13 Sparse Accuracy 0.86% / Best Sparse Accuracy: 0.86%
    Epoch 14 Sparse Accuracy 0.86% / Best Sparse Accurac

## Manual Measurement

In [9]:
import statistics as stat

sparsity=0.9
Eva_final=dict()

Base_model_accuracy=[]
T_base_model=[]
Num_parm_base_model=[]
Base_model_size=[]

Pruned_model_accuracy=[]
T_pruned_model=[]
Num_parm_pruned_model=[]
Pruned_model_size=[]

Pruned_finetune_model_accuracy=[]
T_pruned_finetune_model=[]
Num_parm_pruned_finetune_model=[]
Pruned_finetune_model_size=[]

Spar_conv1_w=[]
Spar_conv2_w=[]
Spar_conv1_lin_w=[]
Spar_conv2_lin_w=[]
Global_spar=[] 


In [10]:
for i in range(5) :
    
        print(f'This is the iteration {i}')
        Eva=dict()

        print(f'Training and evaluation before pruning ')
        model = Net().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)

        for epoch in range(1, 50):
            train(callbacks = None)
            train_acc, test_acc = test(model)
            if epoch % 10==0:
                print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Test: {test_acc:.4f}')

        best_checkpoint = dict()
        best_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
        model.load_state_dict(best_checkpoint['state_dict'])
        recover_model = lambda: model.load_state_dict(best_checkpoint['state_dict'])

        t0=time.time()
        train_acc, base_model_accuracy=test(model)
        t1=time.time()
        t_base_model=t1 - t0
        ###
        base_model_size = get_model_size(model,count_nonzero_only=True)
        num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
        ###   
        print(f"dense model has accuracy on test set={base_model_accuracy:.2f}%")
        print(f"dense model has size={base_model_size/MiB:.2f} MiB")
        print(f"The time inference of base model is ={t_base_model}") 
        print(f"The number of parametrs of base model is:{num_parm_base_model}")   

        #Update my Eva dictionary
        Eva.update({'base model accuracy': base_model_accuracy,
                    'time inference of base model': t_base_model,
                    'number parmameters of base model': num_parm_base_model,
                    'size of base model': base_model_size})


        print('_______________________________________________________')
        print(f'Prune the Model and Re-Evaluate the Accuracy')
        recover_model()

        parameters_to_prune = [
            (model.conv1, 'weight'),
             (model.conv1.lin, 'weight'),
            (model.conv2, 'weight'),
             (model.conv2.lin, 'weight'),
        ]
        #######Global Pruning by Pytorch############################
        global_L1(model,parameters_to_prune,  sparsity)

        ###Sparsities of layyer

        spar_dict=get_model_sparsity(model)  
        Eva.update(spar_dict)



        ##Result
        t0=time.time()
        train_acc, pruned_model_accuracy=test(model)
        t1=time.time()
        t_pruned_model=t1 - t0
        ###
        pruned_model_size = get_model_size(model,count_nonzero_only=True)
        num_parm_pruned_model=get_num_parameters(model, count_nonzero_only=True)
        ###   
        print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_model_accuracy:.2f}%")
        print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB")
        print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_model}") 
        print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_model}")
        print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB, "
              f"which is {base_model_size/pruned_model_size:.2f}X smaller than "
              f"the {base_model_size/MiB:.2f} MiB dense model")



        #Update my Eva dictionary
        Eva.update({'pruned model accuracy': pruned_model_accuracy,
                    'time inference of pruned model': t_pruned_model,
                    'number parmameters of pruned model': num_parm_pruned_model,
                    'size of pruned model': pruned_model_size})



        print('_______________________________________________________')
        print(f'Finetuning Fine-grained Pruned Sparse Model')

        best_sparse_checkpoint = dict()
        best_sparse_accuracy = 0
        num_finetune_epochs=50

        for epoch in range(num_finetune_epochs):
            # At the end of each train iteration, we have to apply the pruning mask
            #    to keep the model sparse during the training
            train(callbacks=[lambda: global_L1(model,parameters_to_prune, sparsity)])
            train_acc, accuracy = test(model)

            is_best = accuracy > best_sparse_accuracy
            if is_best:
                best_sparse_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
                best_sparse_accuracy = accuracy

            if epoch % 10 == 0:
                print(f'Epoch {epoch} Sparse Accuracy {accuracy:.2f}% / Best Sparse Accuracy: {best_sparse_accuracy:.2f}%')

        model.load_state_dict(best_sparse_checkpoint['state_dict'])


        t0=time.time()
        train_acc,pruned_finetune_model_accuracy=test(model)
        t1=time.time()
        t_pruned_finetune_model=t1 - t0
        ###
        pruned_finetune_model_size = get_model_size(model,count_nonzero_only=True)
        num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
        ###   
        print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
        print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
        print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
        print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
        print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
              f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
          f"the {base_model_size/MiB:.2f} MiB dense model")


         #Update my Eva dictionary
        Eva.update({'pruned and finetune model accuracy': pruned_finetune_model_accuracy,
                    'time inference of pruned and finetune model': t_pruned_finetune_model,
                    'number parmameters of pruned and finetune model': num_parm_pruned_finetune_model,
                    'size of pruned and finetune model':  pruned_finetune_model_size})


        Base_model_accuracy.append(Eva['base model accuracy'])
        T_base_model.append(Eva['time inference of base model'])
        Num_parm_base_model.append(int(Eva['number parmameters of base model']))
        Base_model_size.append(int(Eva['size of base model']))

        Pruned_model_accuracy.append(Eva['pruned model accuracy'])
        T_pruned_model.append(Eva['time inference of pruned model'])
        Num_parm_pruned_model.append(int(Eva['number parmameters of pruned model']))
        Pruned_model_size.append(int(Eva['size of pruned model']))

        Pruned_finetune_model_accuracy.append(Eva['pruned and finetune model accuracy'])
        T_pruned_finetune_model.append(Eva['time inference of pruned and finetune model'])
        Num_parm_pruned_finetune_model.append(int(Eva['number parmameters of pruned and finetune model']))
        Pruned_finetune_model_size.append(int(Eva['size of pruned and finetune model']))


        Spar_conv1_w.append(Eva['conv1.weight'])
        Spar_conv1_lin_w.append(Eva['conv1.lin.weight'])
        Spar_conv2_w.append(Eva['conv2.weight'])
        Spar_conv2_lin_w.append(Eva['conv2.lin.weight'])
        Global_spar.append(Eva['Global sparsity']) 
                

This is the iteration 0
Training and evaluation before pruning 
Epoch: 010, Train: 0.9210, Test: 0.8320
Epoch: 020, Train: 0.9526, Test: 0.8580
Epoch: 030, Train: 0.9719, Test: 0.8740
Epoch: 040, Train: 0.9754, Test: 0.8600
dense model has accuracy on test set=0.86%
dense model has size=0.26 MiB
The time inference of base model is =21.40814518928528
The number of parametrs of base model is:69143
_______________________________________________________
Prune the Model and Re-Evaluate the Accuracy
Sparsity in conv1.weight: 0.934
Sparsity in conv1.lin.weight: 0.840
Sparsity in conv2.weight: 0.482
Sparsity in conv2.lin.weight: 0.027
Global sparsity: 0.900
90.0% sparse model has accuracy on test set=0.86%
90.0% sparse model has size=0.03 MiB
The time inference of 90.0% sparse model is =19.585023880004883
The number of parametrs of 90.0% sparse model is:6935
90.0% sparse model has size=0.03 MiB, which is 9.97X smaller than the 0.26 MiB dense model
_____________________________________________

In [11]:
Eva_final=dict()
base_model_accuracy_mean = stat.mean(Base_model_accuracy)
base_model_accuracy_std =  stat.stdev(Base_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)

Eva_final.update({'base model accuracy':float(format(base_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of base model accuracy':float(format(base_model_accuracy_std, '.3f'))})
                 
t_base_model_mean =stat.mean(T_base_model)
t_base_model_std =stat.stdev(T_base_model)  
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of base model':float(format(t_base_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of base model':float(format(t_base_model_std, '.3f'))})


num_parm_base_model_mean = stat.mean(Num_parm_base_model)
num_parm_base_model_std = stat.stdev(Num_parm_base_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of base model':num_parm_base_model_mean})
Eva_final.update({'Std of number parmameters of base model':num_parm_base_model_std})

base_model_size_mean = stat.mean(Base_model_size)
base_model_size_std = stat.stdev(Base_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'base_model_size':base_model_size_mean})
Eva_final.update({'Std of base_model_size':base_model_size_std})

#################################

pruned_model_accuracy_mean =stat.mean(Pruned_model_accuracy)
pruned_model_accuracy_std = stat.stdev(Pruned_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned model accuracy':float(format(pruned_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of pruned model accuracy':float(format(pruned_model_accuracy_std, '.3f'))})
                 

t_pruned_model_mean = stat.mean(T_pruned_model)
t_pruned_model_std =stat.stdev(T_pruned_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of pruned model':float(format(t_pruned_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of pruned model':float(format(t_pruned_model_std, '.3f'))})

num_parm_pruned_model_mean = stat.mean(Num_parm_pruned_model)
num_parm_pruned_model_std = stat.stdev(Num_parm_pruned_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of pruned model':num_parm_pruned_model_mean})
Eva_final.update({'Std of number parmameters of pruned model':num_parm_pruned_model_std})

pruned_model_size_mean =stat.mean( Pruned_model_size)
pruned_model_size_std = stat.stdev(Pruned_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned model size':pruned_model_size_mean})
Eva_final.update({'Std of pruned_model_size':pruned_model_size_std })

#################################
pruned_finetune_model_accuracy_mean =stat.mean(Pruned_finetune_model_accuracy)
pruned_finetune_model_accuracy_std = stat.stdev(Pruned_finetune_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned finetune model accuracy':float(format(pruned_finetune_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of pruned finetune model accuracy':float(format(pruned_finetune_model_accuracy_std, '.3f'))})                 

t_pruned_finetune_model_mean =stat.mean(T_pruned_finetune_model)
t_pruned_finetune_model_std =stat.stdev(T_pruned_finetune_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of pruned finetune model':float(format(t_pruned_finetune_model_mean,'.3f'))})
Eva_final.update({'Std of time inference of pruned finetune model':float(format(t_pruned_finetune_model_std,'.3f'))})

num_parm_pruned_finetune_model_mean =stat.mean(Num_parm_pruned_finetune_model)
num_parm_pruned_finetune_model_std = stat.stdev(Num_parm_pruned_finetune_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of pruned finetune model':num_parm_pruned_finetune_model_mean})
Eva_final.update({'Std of number parmameters of pruned finetune model':num_parm_pruned_finetune_model_std })

pruned_finetune_model_size_mean = stat.mean(Pruned_finetune_model_size)
pruned_finetune_model_size_std = stat.stdev(Pruned_finetune_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned finetune model size':pruned_finetune_model_size_mean})
Eva_final.update({'Std of pruned finetune model size':pruned_finetune_model_size_std})


sparsity_conv1_w_mean = stat.mean(Spar_conv1_w)
sparsity_conv1_w_std = stat.stdev(Spar_conv1_w)
Eva_final.update({'Sparsity in conv1.weight':float(format(sparsity_conv1_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in conv1.weight':float(format(sparsity_conv1_w_std,'.3f'))})

sparsity_conv1_lin_w_mean = stat.mean( Spar_conv1_lin_w)
sparsity_conv1_lin_w_std = stat.stdev( Spar_conv1_lin_w)
Eva_final.update({'Sparsity in conv1.lin.weight':float(format(sparsity_conv1_lin_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in conv1.lin.weight':float(format(sparsity_conv1_lin_w_std,'.3f'))})

sparsity_conv2_w_mean = stat.mean(Spar_conv2_w)
sparsity_conv2_w_std = stat.stdev(Spar_conv2_w)
Eva_final.update({'Sparsity in conv2.weight':float(format(sparsity_conv2_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in conv2.weight':float(format(sparsity_conv2_w_std,'.3f'))})

sparsity_conv2_lin_w_mean = stat.mean( Spar_conv2_lin_w)
sparsity_conv2_lin_w_std = stat.stdev( Spar_conv2_lin_w)
Eva_final.update({'Sparsity in conv2.lin.weight':float(format(sparsity_conv2_lin_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in conv2.lin.weight':float(format(sparsity_conv2_lin_w_std,'.3f'))})

  

Global_sparsity_mean = stat.mean(Global_spar)
Global_sparsity_std = stat.stdev(Global_spar)
Eva_final.update({'Global sparsity':float(format(Global_sparsity_mean,'.3f'))})
Eva_final.update({'Std of Global sparsity':float(format(Global_sparsity_std,'.3f'))})


#################################


print(f"All measurement about pruning process of sparsity:{sparsity*100}% ")   
Eva_final

All measurement about pruning process of sparsity:90.0% 


{'base model accuracy': 0.864,
 'Std of base model accuracy': 0.005,
 'time inference of base model': 17.888,
 'Std of time inference of base model': 1.991,
 'number parmameters of base model': 69143,
 'Std of number parmameters of base model': 0.0,
 'base_model_size': 2212576,
 'Std of base_model_size': 0.0,
 'pruned model accuracy': 0.862,
 'Std of pruned model accuracy': 0.008,
 'time inference of pruned model': 18.788,
 'Std of time inference of pruned model': 1.123,
 'number parmameters of pruned model': 6935,
 'Std of number parmameters of pruned model': 0.0,
 'pruned model size': 221920,
 'Std of pruned_model_size': 0.0,
 'pruned finetune model accuracy': 0.874,
 'Std of pruned finetune model accuracy': 0.003,
 'time inference of pruned finetune model': 17.424,
 'Std of time inference of pruned finetune model': 1.022,
 'number parmameters of pruned finetune model': 6935,
 'Std of number parmameters of pruned finetune model': 0.0,
 'pruned finetune model size': 221920,
 'Std of p